<a href="https://colab.research.google.com/github/toryor31oct/group15-Fun-Rai-Khwam-plod-Phai/blob/Nam-ing/fun_rai_kwam_plod_phai(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
import random
from datetime import datetime, timedelta
import pandas as pd

# โหลดข้อมูลลูกค้าจาก CSV
df_customers = pd.read_csv("customer.csv")


class Policy:
    """Class สำหรับจัดการกรมธรรม์ประกันภัย (แก้ไขเรื่อง Leap Year และ Edge Cases)"""

    PLAN_CONFIGS = {
        "ประกันชีวิต": {
            "base_rate": 0.015,
            "min_coverage": 500000,
            "max_coverage": 3000000,
            "payment_modes": ["รายปี", "รายเดือน"],
        },
        "ประกันสุขภาพ": {
            "base_rate": 0.025,
            "min_coverage": 200000,
            "max_coverage": 1500000,
            "payment_modes": ["รายปี", "รายกึ่งปี", "รายเดือน"],
        },
        "ประกันอุบัติเหตุ": {
            "base_rate": 0.005,
            "min_coverage": 100000,
            "max_coverage": 1000000,
            "payment_modes": ["รายปี"],
        },
        "ประกันโรคร้ายแรง": {
            "base_rate": 0.020,
            "min_coverage": 300000,
            "max_coverage": 2000000,
            "payment_modes": ["รายปี", "รายกึ่งปี"],
        },
    }

    RISK_OCCUPATIONS = {
        "ตำรวจ": 1.35,
        "คนขับรถ": 1.30,
        "เกษตรกร": 1.20,
        "วิศวกร": 1.15,
        "ค้าขาย": 1.05,
        "พนักงานบริษัท": 1.00,
        "แพทย์": 1.00,
        "ข้าราชการ": 0.95,
        "ครู": 0.95,
    }

    def __init__(
        self,
        policy_id,
        customer_id,
        plan_name,
        coverage_amount,
        payment_mode,
        age=30,
        occupation="พนักงานบริษัท",
        start_date=None,
    ):
        self.policy_id = policy_id
        self.customer_id = customer_id
        self.plan_name = plan_name
        self.coverage_amount = coverage_amount
        self.payment_mode = payment_mode
        self.age = age if pd.notnull(age) else 30  # ป้องกันค่า NaN
        self.occupation = (
            occupation if pd.notnull(occupation) else "พนักงานบริษัท"
        )

        # สุ่มวันที่เริ่มต้น
        self.start_date = (
            start_date
            if start_date
            else datetime(2025, 1, 1) + timedelta(days=random.randint(0, 360))
        )
        # ใช้ timedelta(days=365) ป้องกัน Error กรณีตรงกับวันที่ 29 ก.พ.
        self.end_date = self.start_date + timedelta(days=365)
        self.status = "Active"

        self.premium = self.calculate_premium()

    def calculate_premium(self):
        """คำนวณเบี้ยประกันตามปัจจัยความเสี่ยง"""
        base_rate = self.PLAN_CONFIGS[self.plan_name]["base_rate"]

        # ปัจจัยด้านอายุ
        if self.age < 30:
            age_factor = 0.90
        elif self.age <= 50:
            age_factor = 1.10
        else:
            age_factor = 1.35

        # ปัจจัยด้านอาชีพ
        occ_factor = self.RISK_OCCUPATIONS.get(self.occupation, 1.00)

        # เบี้ยประกันฐานต่อปี
        annual_premium = (
            self.coverage_amount * base_rate * age_factor * occ_factor
        )

        # งวดการชำระเงิน
        if self.payment_mode == "รายเดือน":
            premium = (annual_premium / 12) * 1.05
        elif self.payment_mode == "รายกึ่งปี":
            premium = (annual_premium / 2) * 1.02
        else:
            premium = annual_premium

        return round(premium, 2)


# --- จำลองสร้างข้อมูล 300 รายการ ---
random.seed(42)  # ตั้งค่า seed เพื่อให้ได้ผลลัพธ์สุ่มที่ซ้ำเดิมได้ตอนตรวจงาน
policies_list = []

for idx, row in df_customers.iterrows():
    policy_id = f"POL{idx+1:04d}"
    plan_name = random.choice(list(Policy.PLAN_CONFIGS.keys()))
    config = Policy.PLAN_CONFIGS[plan_name]

    coverage = (
        random.randint(
            config["min_coverage"] // 50000, config["max_coverage"] // 50000
        )
        * 50000
    )
    payment_mode = random.choice(config["payment_modes"])

    policy_obj = Policy(
        policy_id=policy_id,
        customer_id=row["Customer_ID"],
        plan_name=plan_name,
        coverage_amount=coverage,
        payment_mode=payment_mode,
        age=row["Age"],
        occupation=row["Occupation"],
    )

    policies_list.append(
        {
            "Policy_ID": policy_obj.policy_id,
            "Customer_ID": policy_obj.customer_id,
            "Plan_Name": policy_obj.plan_name,
            "Coverage_Amount": policy_obj.coverage_amount,
            "Payment_Mode": policy_obj.payment_mode,
            "Premium": policy_obj.premium,
            "Start_Date": policy_obj.start_date.strftime("%Y-%m-%d"),
            "End_Date": policy_obj.end_date.strftime("%Y-%m-%d"),
            "Status": policy_obj.status,
        }
    )

df_policies = pd.DataFrame(policies_list)
df_policies.to_csv("policy.csv", index=False, encoding="utf-8-sig")

print("ตรวจสอบและรันข้อมูลเรียบร้อยแล้ว ได้ไฟล์ policy.csv จำนวน 300 รายการ")

ตรวจสอบและรันข้อมูลเรียบร้อยแล้ว ได้ไฟล์ policy.csv จำนวน 300 รายการ
